# Off-ball Movement Analysis: Offers to Receive

## Purpose
This notebook extends the Field Tilt, Progression Quality, and Final Third Zone Dominance analyses by examining off-ball movement.

## Research Question
How did off-ball movement support territory control and final-third access during the 2026 FIFA World Cup?

## Data Scope
- Source file: `site_official_stats_team_wide_flagged.csv`
- Source: FIFA official Match Centre statistics
- Main filter: `stats_complete == True`
- Excluded match: Belgium vs Egypt (`match_id = 400021478`), because FIFA only provides Live Statistics for that match
- Views: Overall, Group Stage, Knockout Stage

## Key Metrics
- Offers per Match: total offers to receive per team match
- In-behind Offer Share: in-behind offers / total offers x 100
- Between-lines Offer Share: in-between offers / total offers x 100
- In-front Offer Share: in-front offers / total offers x 100
- Reception Access per Match: receptions between midfield/defensive lines + receptions behind the defensive line, per match
- Offer-to-Reception Access Rate: selected line-access receptions / selected line-access offers x 100

## Interpretation Rule
These metrics describe off-ball availability and receiving access. They do not measure player-level movement quality, pass difficulty, opponent structure, or causality by themselves.

## Data Limitations for Publication
- Team match counts differ across the tournament because some teams played 3 matches and finalists played up to 8 matches. Spain-centered rankings are descriptive champion profiling, not a causal model of why Spain won.
- Belgium vs Egypt (`match_id = 400021478`) is excluded from the main analytical tables because FIFA provides only `Live Statistics` for that match. As a result, Belgium has 5 full-stat matches instead of 6, and Egypt has 4 full-stat matches instead of 5. Per-match metrics for those two teams can be slightly inflated because one real match is not in the denominator.




In [ ]:
"""
Step 1: Environment setup and data load
=======================================
Only BASE_DIR should need editing if the project folder moves.
"""

import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from adjustText import adjust_text
except ModuleNotFoundError:
    print("[Setup] adjustText is not installed in this Jupyter kernel. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "adjustText"])
    from adjustText import adjust_text
    print("[Setup] adjustText installed successfully.")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break
DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
PUBLIC_DIR = BASE_DIR
OUTPUT_DATA_DIR = PUBLIC_DIR / "data"
OUTPUT_FIG_DIR = PUBLIC_DIR / "figures"

RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"
GROUP_STAGE_PATH = DATA_DIR / "site_official_stats_team_wide_group_stage.csv"

for path_name, path_value in {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,
    "PUBLIC_DIR": PUBLIC_DIR,
    "OUTPUT_DATA_DIR": OUTPUT_DATA_DIR,
    "OUTPUT_FIG_DIR": OUTPUT_FIG_DIR,
}.items():
    print(f"{path_name}: {path_value}")

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_csv(RAW_CSV_PATH)
group_stage_reference = pd.read_csv(GROUP_STAGE_PATH)

print(f"Raw rows: {len(raw_df):,}")
print(f"Raw matches: {raw_df['match_id'].nunique():,}")
print(f"Raw teams: {raw_df['team_name'].nunique():,}")

display(raw_df.head())






In [ ]:
"""
Step 2: Apply data scope and phase labels
=========================================
Use only matches with full FIFA Official Stats.
"""

df = raw_df[raw_df["stats_complete"] == True].copy()

excluded_matches = raw_df.loc[raw_df["stats_complete"] != True, ["match_id", "team_name", "opponent_name", "stats_source"]].drop_duplicates()
print("Excluded rows because stats_complete is not True:", len(raw_df) - len(df))
if not excluded_matches.empty:
    display(excluded_matches)

group_stage_match_ids = set(group_stage_reference["match_id"].unique())

df["competition_phase"] = np.where(
    df["match_id"].isin(group_stage_match_ids),
    "Group Stage",
    "Knockout Stage",
)

phase_check = (
    df[["match_id", "competition_phase"]]
    .drop_duplicates()
    .groupby("competition_phase")
    .size()
    .reset_index(name="matches")
)

display(phase_check)
print(f"Analysis rows: {len(df):,}")
print(f"Analysis matches: {df['match_id'].nunique():,}")
print(f"Analysis teams: {df['team_name'].nunique():,}")





In [ ]:
"""
Step 3: Create match-team off-ball movement metrics
===================================================
"""

CHAMPION_TEAM = "Spain"

# Semifinalists = teams with the most matches played (8 each): reached
# the final or the third-place match.
SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]

offer_cols = {
    "offers_total": "attacking__offers_to_receive__total",
    "offers_in_behind": "attacking__offers_to_receive__in_behind",
    "offers_between_lines": "attacking__offers_to_receive__in_between",
    "offers_in_front": "attacking__offers_to_receive__in_front",
    "receptions_between_lines": "attacking__offers_to_receive__receptions_between_midfield_and_defensive_lines",
    "receptions_in_behind": "attacking__offers_to_receive__receptions_behind_the_defensive_line",
}

final_third_cols = [
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

required_cols = ["match_id", "team_name", "competition_phase"] + list(offer_cols.values()) + final_third_cols
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan) if isinstance(denominator, pd.Series) else denominator
    return numerator / denominator

team_match = df[["match_id", "team_name", "team_side", "opponent_name", "competition_phase"] + list(offer_cols.values()) + final_third_cols].copy()
team_match = team_match.rename(columns={v: k for k, v in offer_cols.items()})
team_match["final_third_entries_total"] = team_match[final_third_cols].sum(axis=1, min_count=5)
team_match["match_final_third_entries_total"] = team_match.groupby("match_id")["final_third_entries_total"].transform("sum")
team_match["opponent_final_third_entries_total"] = team_match["match_final_third_entries_total"] - team_match["final_third_entries_total"]
team_match["field_tilt_proxy"] = safe_divide(team_match["final_third_entries_total"], team_match["match_final_third_entries_total"]) * 100
team_match["line_access_offers"] = team_match["offers_in_behind"] + team_match["offers_between_lines"]
team_match["line_access_receptions"] = team_match["receptions_in_behind"] + team_match["receptions_between_lines"]

team_match["in_behind_offer_share"] = safe_divide(team_match["offers_in_behind"], team_match["offers_total"]) * 100
team_match["between_lines_offer_share"] = safe_divide(team_match["offers_between_lines"], team_match["offers_total"]) * 100
team_match["in_front_offer_share"] = safe_divide(team_match["offers_in_front"], team_match["offers_total"]) * 100
team_match["offer_to_reception_access_rate"] = safe_divide(team_match["line_access_receptions"], team_match["line_access_offers"]) * 100
team_match["between_lines_reception_rate"] = safe_divide(team_match["receptions_between_lines"], team_match["offers_between_lines"]) * 100
team_match["in_behind_reception_rate"] = safe_divide(team_match["receptions_in_behind"], team_match["offers_in_behind"]) * 100

print("Team-match metric rows:", len(team_match))
display(team_match.head())






In [ ]:
"""
Step 4: Aggregate to team-level metrics by competition phase
============================================================
"""

sum_cols = [
    "offers_total",
    "offers_in_behind",
    "offers_between_lines",
    "offers_in_front",
    "receptions_between_lines",
    "receptions_in_behind",
    "line_access_offers",
    "line_access_receptions",
    "final_third_entries_total",
    "opponent_final_third_entries_total",
]


def build_team_phase_metrics(source_df, phase_name):
    grouped = (
        source_df
        .groupby("team_name", as_index=False)
        .agg(
            matches=("match_id", "nunique"),
            **{col: (col, "sum") for col in sum_cols},
        )
    )
    grouped["competition_phase"] = phase_name
    grouped["field_tilt_proxy"] = safe_divide(
        grouped["final_third_entries_total"],
        grouped["final_third_entries_total"] + grouped["opponent_final_third_entries_total"]
    ) * 100
    grouped["offers_per_match"] = grouped["offers_total"] / grouped["matches"]
    grouped["in_behind_offers_per_match"] = grouped["offers_in_behind"] / grouped["matches"]
    grouped["between_lines_offers_per_match"] = grouped["offers_between_lines"] / grouped["matches"]
    grouped["in_front_offers_per_match"] = grouped["offers_in_front"] / grouped["matches"]
    grouped["receptions_between_lines_per_match"] = grouped["receptions_between_lines"] / grouped["matches"]
    grouped["receptions_in_behind_per_match"] = grouped["receptions_in_behind"] / grouped["matches"]
    grouped["reception_access_per_match"] = grouped["line_access_receptions"] / grouped["matches"]
    grouped["final_third_entries_per_match"] = grouped["final_third_entries_total"] / grouped["matches"]

    grouped["in_behind_offer_share"] = safe_divide(grouped["offers_in_behind"], grouped["offers_total"]) * 100
    grouped["between_lines_offer_share"] = safe_divide(grouped["offers_between_lines"], grouped["offers_total"]) * 100
    grouped["in_front_offer_share"] = safe_divide(grouped["offers_in_front"], grouped["offers_total"]) * 100
    grouped["offer_to_reception_access_rate"] = safe_divide(grouped["line_access_receptions"], grouped["line_access_offers"]) * 100
    grouped["between_lines_reception_rate"] = safe_divide(grouped["receptions_between_lines"], grouped["offers_between_lines"]) * 100
    grouped["in_behind_reception_rate"] = safe_divide(grouped["receptions_in_behind"], grouped["offers_in_behind"]) * 100

    return grouped

phase_frames = []
for phase_name, phase_source in team_match.groupby("competition_phase"):
    phase_frames.append(build_team_phase_metrics(phase_source, phase_name))
phase_frames.append(build_team_phase_metrics(team_match, "Overall"))

off_ball_metrics_by_phase = pd.concat(phase_frames, ignore_index=True)

ordered_cols = [
    "competition_phase",
    "team_name",
    "matches",
    "field_tilt_proxy",
    "final_third_entries_per_match",
    "offers_per_match",
    "in_behind_offers_per_match",
    "between_lines_offers_per_match",
    "in_front_offers_per_match",
    "in_behind_offer_share",
    "between_lines_offer_share",
    "in_front_offer_share",
    "receptions_in_behind_per_match",
    "receptions_between_lines_per_match",
    "reception_access_per_match",
    "offer_to_reception_access_rate",
    "in_behind_reception_rate",
    "between_lines_reception_rate",
] + sum_cols

off_ball_metrics_by_phase = off_ball_metrics_by_phase[ordered_cols]

display(off_ball_metrics_by_phase.head())
print(off_ball_metrics_by_phase.groupby("competition_phase")["team_name"].nunique())






In [ ]:
"""
Step 5: Save off-ball movement data outputs
===========================================
"""

off_ball_metrics_path = OUTPUT_DATA_DIR / "off_ball_movement_team_metrics_by_phase.csv"
off_ball_overall_path = OUTPUT_DATA_DIR / "off_ball_movement_team_metrics_overall.csv"
off_ball_spain_path = OUTPUT_DATA_DIR / "off_ball_movement_spain_profile_overall.csv"

off_ball_metrics_by_phase.to_csv(off_ball_metrics_path, index=False)
off_ball_metrics_by_phase[off_ball_metrics_by_phase["competition_phase"] == "Overall"].to_csv(off_ball_overall_path, index=False)
off_ball_metrics_by_phase[
    (off_ball_metrics_by_phase["competition_phase"] == "Overall")
    & (off_ball_metrics_by_phase["team_name"] == CHAMPION_TEAM)
].to_csv(off_ball_spain_path, index=False)

print(f"Saved: {off_ball_metrics_path}")
print(f"Saved: {off_ball_overall_path}")
print(f"Saved: {off_ball_spain_path}")





## Visualization 1: Spain Off-ball Movement Ranking Profile

This chart places Spain's off-ball movement indicators in tournament context. Percentile rank is descriptive: higher means Spain ranked closer to the top of the selected phase for that metric.

In [ ]:
"""
Step 6: Build Spain off-ball movement percentile rankings
=========================================================
"""

plot_phase = "Overall"
# plot_phase = "Group Stage"
# plot_phase = "Knockout Stage"

ranking_metric_map = {
    "Offers per Match": "offers_per_match",
    "Final Third Entries per Match": "final_third_entries_per_match",
    "Field Tilt Proxy": "field_tilt_proxy",
    "Reception Access per Match": "reception_access_per_match",
    "Offer-to-Reception Access Rate": "offer_to_reception_access_rate",
    "In-behind Offer Share": "in_behind_offer_share",
    "Between-lines Offer Share": "between_lines_offer_share",
    "In-front Offer Share": "in_front_offer_share",
    "In-behind Receptions per Match": "receptions_in_behind_per_match",
    "Between-lines Receptions per Match": "receptions_between_lines_per_match",
}

phase_rank_df = off_ball_metrics_by_phase[off_ball_metrics_by_phase["competition_phase"] == plot_phase].copy()
ranking_rows = []
for metric_label, metric_col in ranking_metric_map.items():
    metric_values = phase_rank_df[["team_name", metric_col]].dropna().copy()
    metric_values["rank"] = metric_values[metric_col].rank(method="min", ascending=False)
    teams_in_phase = metric_values["team_name"].nunique()
    metric_values["percentile_rank"] = (teams_in_phase - metric_values["rank"] + 1) / teams_in_phase * 100
    spain_metric = metric_values[metric_values["team_name"] == CHAMPION_TEAM].iloc[0]
    ranking_rows.append({
        "competition_phase": plot_phase,
        "metric_label": metric_label,
        "metric_col": metric_col,
        "value": spain_metric[metric_col],
        "rank": int(spain_metric["rank"]),
        "teams_in_phase": int(teams_in_phase),
        "percentile_rank": spain_metric["percentile_rank"],
    })

spain_off_ball_rankings = pd.DataFrame(ranking_rows).sort_values("percentile_rank", ascending=False)
spain_off_ball_rankings_path = OUTPUT_DATA_DIR / f"off_ball_movement_spain_rankings_{plot_phase.lower().replace(' ', '_')}.csv"
spain_off_ball_rankings.to_csv(spain_off_ball_rankings_path, index=False)

display(spain_off_ball_rankings.round(2))
print(f"Saved: {spain_off_ball_rankings_path}")





In [ ]:
"""
Step 7: Plot Spain off-ball movement percentile rankings
========================================================
"""

plot_df = spain_off_ball_rankings.copy().sort_values("percentile_rank", ascending=True)

fig, ax = plt.subplots(figsize=(11.5, 7.3), dpi=150)
fig.patch.set_facecolor("#f8fafc")
ax.set_facecolor("#f8fafc")

# Soft performance bands.
ax.axvspan(0, 25, color="#fee2e2", alpha=0.28, zorder=0)
ax.axvspan(25, 50, color="#fef3c7", alpha=0.30, zorder=0)
ax.axvspan(50, 75, color="#dbeafe", alpha=0.24, zorder=0)
ax.axvspan(75, 100, color="#dcfce7", alpha=0.26, zorder=0)

colors = np.where(
    plot_df["percentile_rank"] >= 75,
    "#047857",
    np.where(plot_df["percentile_rank"] >= 50, "#0ea5e9", np.where(plot_df["percentile_rank"] >= 25, "#f59e0b", "#dc2626")),
)

bars = ax.barh(
    plot_df["metric_label"],
    plot_df["percentile_rank"],
    color=colors,
    alpha=0.92,
    height=0.64,
)

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    x = row["percentile_rank"]
    label = f"#{int(row['rank'])} / {int(row['teams_in_phase'])}"
    if x >= 88:
        ax.text(x - 2.0, bar.get_y() + bar.get_height() / 2, label, va="center", ha="right", fontsize=8.7, color="white")
    else:
        ax.text(x + 1.4, bar.get_y() + bar.get_height() / 2, label, va="center", ha="left", fontsize=8.7, color="#111827")

ax.axvline(50, linestyle="--", color="#64748b", linewidth=1.0, alpha=0.85)
ax.text(50, len(plot_df) - 0.25, "Median", ha="center", va="bottom", fontsize=8.5, color="#64748b")

ax.set_xlim(0, 104)
ax.set_xlabel("Percentile Rank within Phase (higher = closer to #1)", fontsize=10.5, color="#111827")
ax.set_title(f"Spain Off-ball Movement Profile ({plot_phase})", fontsize=15.5, pad=14, color="#111827")
ax.grid(axis="x", alpha=0.20)
ax.tick_params(axis="both", labelsize=9.5, colors="#111827")

for spine in ax.spines.values():
    spine.set_visible(False)

footnote = (
    "Percentile rankings are descriptive and do not imply causality.\n"
    "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
)
fig.text(0.08, 0.025, footnote, fontsize=7.8, color="#667085", linespacing=1.18)

plt.tight_layout(rect=[0, 0.105, 1, 1])

output_path = OUTPUT_FIG_DIR / f"15_off_ball_movement_spain_profile_{plot_phase.lower().replace(' ', '_')}.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved figure: {output_path}")






## Visualization 2: In-behind vs Between-lines Movement

This scatter plot separates two kinds of off-ball movement: runs behind the defensive line and offers between the lines. Spain is highlighted against the tournament distribution.

In [ ]:
"""
Step 8: Plot In-behind vs Between-lines Offer Share
===================================================
"""

plot_phase = "Overall"
plot_df = off_ball_metrics_by_phase[off_ball_metrics_by_phase["competition_phase"] == plot_phase].copy()

x_col = "in_behind_offer_share"
y_col = "between_lines_offer_share"
color_col = "reception_access_per_match"

fig, ax = plt.subplots(figsize=(14.8, 10.0), dpi=150)
fig.patch.set_facecolor("#f8fafc")
ax.set_facecolor("#f8fafc")

scatter = ax.scatter(
    plot_df[x_col],
    plot_df[y_col],
    c=plot_df[color_col],
    cmap="viridis",
    s=58,
    alpha=0.72,
    edgecolor="white",
    linewidth=0.75,
)

x_median = plot_df[x_col].median()
y_median = plot_df[y_col].median()
ax.axvline(x_median, linestyle="--", color="#64748b", linewidth=1, alpha=0.75)
ax.axhline(y_median, linestyle="--", color="#64748b", linewidth=1, alpha=0.75)

missing_sf = [t for t in SEMIFINALISTS if t not in plot_df["team_name"].values]
if missing_sf:
    raise ValueError(f"Semifinalist teams missing from data: {missing_sf}")

semifinalist_rows = plot_df[plot_df["team_name"].isin(SEMIFINALISTS)]
ax.scatter(
    semifinalist_rows[x_col],
    semifinalist_rows[y_col],
    s=170,
    facecolors="none",
    edgecolors="#dc2626",
    linewidth=2.2,
    zorder=5,
)
texts = []
for _, row in plot_df.sort_values("team_name").iterrows():
    is_sf = row["team_name"] in SEMIFINALISTS
    texts.append(
        ax.text(
            row[x_col],
            row[y_col],
            row["team_name"],
            fontsize=8.8 if is_sf else 7.5,
            fontweight="bold" if is_sf else "normal",
            color="#7f1d1d" if is_sf else "#334155",
            zorder=6 if is_sf else 4,
        )
    )

adjust_text(
    texts,
    ax=ax,
    expand=(1.18, 1.30),
    force_text=(0.32, 0.44),
    force_static=(0.18, 0.26),
    force_pull=(0.012, 0.020),
    max_move=(24, 24),
    min_arrow_len=6,
    arrowprops=dict(arrowstyle="-", color="#94a3b8", lw=0.42, alpha=0.50),
    iter_lim=600,
)

cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
cbar.set_label("Reception Access per Match", fontsize=9.5)
cbar.ax.tick_params(labelsize=8.5)
cbar.outline.set_visible(False)

ax.set_xlabel("In-behind Offer Share (%)", fontsize=10.5)
ax.set_ylabel("Between-lines Offer Share (%)", fontsize=10.5)
ax.set_title(f"In-behind vs Between-lines Offers ({plot_phase})", fontsize=15.5, pad=14)
ax.grid(alpha=0.22)
ax.margins(x=0.08, y=0.10)

for spine in ax.spines.values():
    spine.set_visible(False)


from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="none", markeredgecolor="#dc2626",
           markeredgewidth=2.0, markersize=10, label="Semifinalists (Spain, Argentina, France, England)"),
]
ax.legend(handles=legend_handles, loc="upper left", fontsize=8.5, frameon=True, facecolor="#f8fafc", edgecolor="#d1d5db")

footnote = (
    "Each dot represents one team. Labels are automatically adjusted to reduce overlap. Shares are calculated from team tournament totals in the selected phase.\n"
    "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
)
fig.text(0.08, 0.025, footnote, fontsize=7.8, color="#667085", linespacing=1.18)

plt.tight_layout(rect=[0, 0.105, 1, 1])

output_path = OUTPUT_FIG_DIR / f"16_off_ball_in_behind_vs_between_lines_{plot_phase.lower().replace(' ', '_')}.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved figure: {output_path}")








## Visualization 3: Offer Volume vs Reception Access

This plot checks whether teams that created more off-ball availability also received more often between or behind defensive lines.

In [ ]:
"""
Step 9: Plot Offer Volume vs Reception Access
=============================================
"""

plot_phase = "Overall"
plot_df = off_ball_metrics_by_phase[off_ball_metrics_by_phase["competition_phase"] == plot_phase].copy()

x_col = "offers_per_match"
y_col = "reception_access_per_match"
color_col = "field_tilt_proxy"

fig, ax = plt.subplots(figsize=(14.8, 10.0), dpi=150)
fig.patch.set_facecolor("#f8fafc")
ax.set_facecolor("#f8fafc")

scatter = ax.scatter(
    plot_df[x_col],
    plot_df[y_col],
    c=plot_df[color_col],
    cmap="mako" if "mako" in plt.colormaps() else "viridis",
    s=60,
    alpha=0.75,
    edgecolor="white",
    linewidth=0.75,
)

x_median = plot_df[x_col].median()
y_median = plot_df[y_col].median()
ax.axvline(x_median, linestyle="--", color="#64748b", linewidth=1, alpha=0.75)
ax.axhline(y_median, linestyle="--", color="#64748b", linewidth=1, alpha=0.75)

missing_sf = [t for t in SEMIFINALISTS if t not in plot_df["team_name"].values]
if missing_sf:
    raise ValueError(f"Semifinalist teams missing from data: {missing_sf}")

semifinalist_rows = plot_df[plot_df["team_name"].isin(SEMIFINALISTS)]
ax.scatter(
    semifinalist_rows[x_col],
    semifinalist_rows[y_col],
    s=170,
    facecolors="none",
    edgecolors="#dc2626",
    linewidth=2.2,
    zorder=5,
)
texts = []
for _, row in plot_df.sort_values("team_name").iterrows():
    is_sf = row["team_name"] in SEMIFINALISTS
    texts.append(
        ax.text(
            row[x_col],
            row[y_col],
            row["team_name"],
            fontsize=8.8 if is_sf else 7.5,
            fontweight="bold" if is_sf else "normal",
            color="#7f1d1d" if is_sf else "#334155",
            zorder=6 if is_sf else 4,
        )
    )

adjust_text(
    texts,
    ax=ax,
    expand=(1.18, 1.30),
    force_text=(0.32, 0.44),
    force_static=(0.18, 0.26),
    force_pull=(0.012, 0.020),
    max_move=(24, 24),
    min_arrow_len=6,
    arrowprops=dict(arrowstyle="-", color="#94a3b8", lw=0.42, alpha=0.50),
    iter_lim=600,
)

cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
cbar.set_label("Field Tilt Proxy (%)", fontsize=9.5)
cbar.ax.tick_params(labelsize=8.5)
cbar.outline.set_visible(False)

ax.set_xlabel("Offers to Receive per Match", fontsize=10.5)
ax.set_ylabel("Reception Access per Match", fontsize=10.5)
ax.set_title(f"Offer Volume vs Reception Access ({plot_phase})", fontsize=15.5, pad=14)
ax.grid(alpha=0.22)
ax.margins(x=0.08, y=0.10)

for spine in ax.spines.values():
    spine.set_visible(False)


from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="none", markeredgecolor="#dc2626",
           markeredgewidth=2.0, markersize=10, label="Semifinalists (Spain, Argentina, France, England)"),
]
ax.legend(handles=legend_handles, loc="upper left", fontsize=8.5, frameon=True, facecolor="#f8fafc", edgecolor="#d1d5db")

footnote = (
    "Each dot represents one team. Labels are automatically adjusted to reduce overlap. Reception Access = receptions between midfield/defensive lines + receptions behind the defensive line.\n"
    "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
)
fig.text(0.08, 0.025, footnote, fontsize=7.8, color="#667085", linespacing=1.18)

plt.tight_layout(rect=[0, 0.105, 1, 1])

output_path = OUTPUT_FIG_DIR / f"17_off_ball_offer_volume_vs_reception_access_{plot_phase.lower().replace(' ', '_')}.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved figure: {output_path}")








## Visualization 4: Spain vs Tournament Average Movement Mix

This chart compares Spain's offer profile with the tournament average. It shows whether Spain's off-ball movement skewed more toward receiving behind, between, or in front of defensive lines.

In [ ]:
"""
Step 10: Plot Spain vs Tournament Average Offer Mix
===================================================
"""

plot_phase = "Overall"
plot_df = off_ball_metrics_by_phase[off_ball_metrics_by_phase["competition_phase"] == plot_phase].copy()

mix_metrics = [
    ("In-behind", "in_behind_offer_share"),
    ("Between-lines", "between_lines_offer_share"),
    ("In-front", "in_front_offer_share"),
]

spain = plot_df[plot_df["team_name"] == CHAMPION_TEAM].iloc[0]
tournament_avg = plot_df[[col for _, col in mix_metrics]].mean()

mix_df = pd.DataFrame({
    "Movement Type": [label for label, _ in mix_metrics],
    "Spain": [spain[col] for _, col in mix_metrics],
    "Tournament Average": [tournament_avg[col] for _, col in mix_metrics],
})

x = np.arange(len(mix_df))
width = 0.34

fig, ax = plt.subplots(figsize=(9.8, 6.4), dpi=150)
fig.patch.set_facecolor("#f8fafc")
ax.set_facecolor("#f8fafc")

bars_spain = ax.bar(x - width / 2, mix_df["Spain"], width, label="Spain", color="#047857", alpha=0.92)
bars_avg = ax.bar(x + width / 2, mix_df["Tournament Average"], width, label="Tournament Average", color="#94a3b8", alpha=0.82)

for bars in [bars_spain, bars_avg]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.7,
            f"{height:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8.7,
            color="#111827",
        )

ax.set_xticks(x)
ax.set_xticklabels(mix_df["Movement Type"], fontsize=10)
ax.set_ylabel("Share of Total Offers (%)", fontsize=10.5)
ax.set_title(f"Spain vs Tournament Average Off-ball Movement Mix ({plot_phase})", fontsize=15.0, pad=14)
ax.legend(frameon=False, fontsize=9.5, loc="upper right")
ax.grid(axis="y", alpha=0.22)
ax.set_ylim(0, max(mix_df[["Spain", "Tournament Average"]].max()) + 8)

for spine in ax.spines.values():
    spine.set_visible(False)

footnote = (
    "Movement mix is based on FIFA Offers to Receive categories. Values are team-level shares from tournament totals.\n"
    "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
)
fig.text(0.08, 0.025, footnote, fontsize=7.8, color="#667085", linespacing=1.18)

plt.tight_layout(rect=[0, 0.105, 1, 1])

output_path = OUTPUT_FIG_DIR / f"18_off_ball_spain_vs_tournament_average_offer_mix_{plot_phase.lower().replace(' ', '_')}.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved figure: {output_path}")
display(mix_df.round(2))






In [ ]:
"""
Step 11: Draft notes for report interpretation
==============================================
Use these notes as a starting point for the public report.
"""

spain_summary = off_ball_metrics_by_phase[
    (off_ball_metrics_by_phase["competition_phase"] == "Overall")
    & (off_ball_metrics_by_phase["team_name"] == CHAMPION_TEAM)
].iloc[0]

print("Off-ball Movement Draft Notes")
print("================================")
print(f"Spain offers per match: {spain_summary['offers_per_match']:.1f}")
print(f"Spain in-behind offer share: {spain_summary['in_behind_offer_share']:.1f}%")
print(f"Spain between-lines offer share: {spain_summary['between_lines_offer_share']:.1f}%")
print(f"Spain in-front offer share: {spain_summary['in_front_offer_share']:.1f}%")
print(f"Spain reception access per match: {spain_summary['reception_access_per_match']:.1f}")
print(f"Spain offer-to-reception access rate: {spain_summary['offer_to_reception_access_rate']:.1f}%")

print("\nSuggested interpretation frame:")
print(
    "This section connects Spain's territorial and progression profile to off-ball availability. "
    "Offers to Receive show where passing options were created before the ball arrived, while reception access checks whether those movements turned into actual receiving positions between or behind defensive lines. "
    "The analysis should be read descriptively: it identifies tendencies in Spain's movement profile compared with the tournament, not a causal explanation of winning."
)





## Visualization 5: Semifinalists vs Rest — Off-ball Movement Volume

This chart tests whether the semifinalists (Spain, Argentina, France, England) also separated from the rest of the field on off-ball movement volume, not just territory and progression.

In [ ]:
"""
Step 10: Semifinalists vs rest — off-ball movement volume comparison
=======================================================================
"""

overall_df = off_ball_metrics_by_phase[
    off_ball_metrics_by_phase["competition_phase"] == "Overall"
].copy()

missing_sf = [t for t in SEMIFINALISTS if t not in overall_df["team_name"].values]
if missing_sf:
    raise ValueError(f"Semifinalist teams missing from data: {missing_sf}")

compare_metrics = ["offers_per_match", "reception_access_per_match"]

overall_df["group"] = np.where(
    overall_df["team_name"].isin(SEMIFINALISTS), "Semifinalists (4)", "Rest of Field (44)"
)

group_comparison = overall_df.groupby("group")[compare_metrics].mean().round(2)
print("[Semifinalists vs Rest — mean values]")
display(group_comparison)

gap_pct = (
    (group_comparison.loc["Semifinalists (4)"] - group_comparison.loc["Rest of Field (44)"])
    / group_comparison.loc["Rest of Field (44)"] * 100
)
print("\n[Gap: Semifinalists vs Rest, %]")
display(gap_pct.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
labels = {"offers_per_match": "Offers to Receive per Match", "reception_access_per_match": "Reception Access per Match"}

for ax, metric in zip(axes, compare_metrics):
    means = group_comparison[metric]
    colors = ["#dc2626", "#94A3B8"]
    bars = ax.bar(means.index, means.values, color=colors, width=0.55)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + max(means.values)*0.02, f"{val:.1f}", ha="center", fontsize=11, fontweight="bold")
    ax.set_title(labels[metric], fontsize=12)
    ax.set_ylim(0, max(means.values) * 1.25)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Semifinalists Moved More Off the Ball, Too", fontsize=15, y=1.03)
fig.text(0.5, -0.02,
    "Semifinalists = Spain, Argentina, France, England (most matches played, 8 each).\n"
    "Source: FIFA Match Centre | Full Official Stats only.",
    ha="center", fontsize=8, color="gray")
plt.tight_layout()

semifinalist_offball_chart_path = OUTPUT_DATA_DIR / "semifinalists_vs_rest_off_ball_volume.png"
plt.savefig(semifinalist_offball_chart_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {semifinalist_offball_chart_path}")